# Double Responses in Decision Tasks - Full Pipeline
Group 17
## 1. Model & Priors

In [2]:
import sys
sys.path.insert(0, '../../src')
from priors import prior

for _ in range(3):
    print(prior())

{'nu': array([1.57722685, 0.8259513 ]), 'alpha1': 1.2371612415922888, 'alpha2': np.float64(7.040398264159797), 'tau': 0.1367541617600175}
{'nu': array([2.09773084, 2.93460097]), 'alpha1': 0.8968881786397989, 'alpha2': np.float64(11.349864354183426), 'tau': 0.14000912331290996}
{'nu': array([2.19521439, 1.85484722]), 'alpha1': 0.8365297499773773, 'alpha2': np.float64(15.890714590645498), 'tau': 0.12141051945009165}


## 2. Simulator

In [6]:
from simulator import simulate_dataset

p = prior()
data = simulate_dataset(p['nu'], p['alpha1'], p['alpha2'], p['tau'])
print("double response rate:", data['double_flag'].mean())

double response rate: 0.0


## 3. BayesFlow Simulator + Adapter

In [7]:
from bf_simulator import simulator
from adapter import adapter

batch = simulator.sample(5)
adapted = adapter(batch)
for k, v in adapted.items():
    print(k, v.shape)

INFO:matplotlib.font_manager:generated new fontManager
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
INFO:bayesflow:Using backend 'jax'


n (5, 1)
inference_variables (5, 5)
summary_variables (5, 1000, 4)


/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Training

In [8]:
from load_model import get_posterior
print("trained model loaded")

trained model loaded


## 5. Parameter Recovery Check

In [9]:
from priors import prior
from simulator import simulate_dataset
import numpy as np

true_params = prior()
trial_data = simulate_dataset(
    true_params["nu"], true_params["alpha1"], true_params["alpha2"], true_params["tau"]
)

fake_data = {
    "n": np.array([[1000]]),
    "nu": true_params["nu"].reshape(1, -1),
    "alpha1": np.array([[true_params["alpha1"]]]),
    "alpha2": np.array([[true_params["alpha2"]]]),
    "tau": np.array([[true_params["tau"]]]),
    "first_response_time": trial_data["first_response_time"].reshape(1, -1),
    "correct": trial_data["correct"].reshape(1, -1),
    "double_flag": trial_data["double_flag"].reshape(1, -1),
    "second_response_time": trial_data["second_response_time"].reshape(1, -1),
}

posterior = get_posterior(fake_data)
print("true:", true_params)
print("estimated:", {k: posterior[k].mean(axis=1) for k in ["nu", "alpha1", "alpha2", "tau"]})

Sampling: 100%|██████████| 1/1 [00:01<00:00,  1.32s/batch]

true: {'nu': array([2.93780968, 3.80564558]), 'alpha1': 0.252721143135548, 'alpha2': np.float64(14.276771254587565), 'tau': 0.025745553612013698}
estimated: {'nu': array([[2.76293751, 4.12123274]]), 'alpha1': array([[0.2362143]]), 'alpha2': array([[15.31922192]]), 'tau': array([[0.03519632]])}


## 6. Diagnostics (calibration, SBC)

In [10]:
# TODO (diagnostics pair): calibration plots, SBC, posterior contraction
# use get_posterior() from load_model.py

## 7. Real Data Fitting (Evans et al. 2020)

In [11]:
# TODO (diagnostics pair): load real dataset, run get_posterior() on it,
# compare estimated parameters to results reported in Evans et al. (2020)

## 8. Results Summary / TL;DR

In [12]:
# TODO: final figures + summary for presentation